<a href="https://colab.research.google.com/github/Shesha-Sai-999/knn-language-classifier/blob/main/knn_language_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import io
import random
from collections import defaultdict
from google.colab import files

# Distance functions
def levenshtein_distance(s1, s2):
    if len(s1) == 0:
        return len(s2)
    if len(s2) == 0:
        return len(s1)

    s1 = s1.lower()
    s2 = s2.lower()

    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i-1] == s2[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = min(dp[i-1][j] + 1, dp[i][j-1] + 1, dp[i-1][j-1] + 1)

    raw_dist = dp[m][n]
    return raw_dist / max(len(s1), len(s2))

def extract_features(code):
    code = code.lower()
    features = {
        'has_def': 'def ' in code,
        'has_class': 'class ' in code,
        'has_import': 'import ' in code or 'from ' in code,
        'has_cout': 'cout' in code or '<<' in code,
        'has_vector': 'vector<' in code or 'vector ' in code,
        'has_html': '<html' in code or '<div' in code or '<p>' in code,
        'has_php': '<?php' in code,
        'has_print': 'print(' in code or 'print ' in code,
        'has_for': 'for ' in code,
        'has_while': 'while ' in code
    }
    return features

def preprocess(code):
    code = code.lower()
    code = code.strip()
    return code

def split_dataset(df, train_ratio=0.8):
    df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)
    train_size = int(len(df_shuffled) * train_ratio)
    train_df = df_shuffled[:train_size]
    test_df = df_shuffled[train_size:]
    return train_df, test_df

class CodeLanguageDetector:
    def __init__(self, k=7):
        self.k = k
        self.samples = []
        self.labels = []
        self.features = []

    def train(self, X, y):
        self.samples = X
        self.labels = y
        self.features = [extract_features(code) for code in X]

    def predict_with_neighbors(self, new_code):
        new_code = preprocess(new_code)
        new_features = extract_features(new_code)

        distances = []
        for i, (code, label) in enumerate(zip(self.samples, self.labels)):
            code = preprocess(code)
            lev_dist = levenshtein_distance(new_code, code)

            feature_score = 0
            for feat_name in new_features:
                if new_features[feat_name] == self.features[i].get(feat_name, False):
                    feature_score += 0.05

            total_dist = lev_dist - feature_score
            distances.append((total_dist, label, code))

        distances.sort(key=lambda x: x[0])
        neighbors = distances[:self.k]

        votes = {}
        neighbor_info = []
        for dist, lang, code_snippet in neighbors:
            weight = 1 / (dist + 0.1)
            votes[lang] = votes.get(lang, 0) + weight
            neighbor_info.append((lang, dist, code_snippet[:40]))

        predicted = max(votes, key=votes.get)
        return predicted, neighbor_info

    def predict(self, new_code):
        predicted, _ = self.predict_with_neighbors(new_code)
        return predicted

    def evaluate(self, X_test, y_test):
        matrix = defaultdict(lambda: defaultdict(int))
        predictions = []

        for i, code in enumerate(X_test):
            pred, neighbors = self.predict_with_neighbors(code)
            predictions.append(pred)
            matrix[y_test[i]][pred] += 1

        accuracy = sum(1 for p, a in zip(predictions, y_test) if p == a) / len(y_test)

        languages = list(set(y_test))
        precision = {}
        recall = {}
        f1 = {}

        for lang in languages:
            tp = matrix[lang][lang]
            fp = sum(matrix[other][lang] for other in languages if other != lang)
            fn = sum(matrix[lang][other] for other in languages if other != lang)

            precision[lang] = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall[lang] = tp / (tp + fn) if (tp + fn) > 0 else 0
            f1[lang] = 2 * (precision[lang] * recall[lang]) / (precision[lang] + recall[lang]) if (precision[lang] + recall[lang]) > 0 else 0

        return accuracy, predictions, matrix, precision, recall, f1

print("Upload your CSV file")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[file_name]))

print(f"\nLoaded {len(df)} samples")
print("\nLanguage distribution:")
for lang, count in df['language'].value_counts().items():
    print(f"  {lang}: {count}")

train_df, test_df = split_dataset(df, train_ratio=0.8)

X_train = train_df['code'].tolist()
y_train = train_df['language'].tolist()
X_test = test_df['code'].tolist()
y_test = test_df['language'].tolist()

print(f"\nTraining: {len(X_train)} samples")
print(f"Testing: {len(X_test)} samples")

print("\nFinding optimal K value...")
k_values = range(1, 16, 2)
k_accuracies = []

for k in k_values:
    model = CodeLanguageDetector(k=k)
    model.train(X_train, y_train)
    acc, _, _, _, _, _ = model.evaluate(X_test, y_test)
    k_accuracies.append(acc)
    print(f"  K={k}: {acc*100:.1f}%")

best_k = k_values[k_accuracies.index(max(k_accuracies))]
best_accuracy = max(k_accuracies)

print(f"\nBest K: {best_k} with {best_accuracy*100:.1f}% accuracy")

print("\nTraining final model...")
final_model = CodeLanguageDetector(k=best_k)
final_model.train(X_train, y_train)

accuracy, predictions, conf_matrix, precision, recall, f1 = final_model.evaluate(X_test, y_test)

print("\n" + "="*50)
print("Model Performance")
print("="*50)
print(f"Accuracy: {accuracy*100:.2f}%")
print("\nPer-language metrics:")
for lang in sorted(precision.keys()):
    print(f"  {lang}:")
    print(f"    Precision: {precision[lang]*100:.1f}%")
    print(f"    Recall:    {recall[lang]*100:.1f}%")
    print(f"    F1-Score:  {f1[lang]*100:.1f}%")

print("\nConfusion Matrix (first 5x5):")
conf_items = list(conf_matrix.items())[:5]
for actual, preds in conf_items:
    print(f"  Actual {actual}: {dict(preds)}")

print("\nSample predictions on test data (first 10):")
for i in range(min(10, len(X_test))):
    pred, neighbors = final_model.predict_with_neighbors(X_test[i])
    status = "✓" if pred == y_test[i] else "✗"
    code_preview = X_test[i][:40].replace('\n', ' ')
    print(f"{status} Pred: {pred:8} | Actual: {y_test[i]:8} | {code_preview}...")

    if status == "✗":
        print(f"     Nearest neighbors:")
        for lang, dist, snippet in neighbors[:3]:
            print(f"       - {lang} (dist={dist:.3f}): {snippet}...")

print("\n" + "="*50)
print("Live Demo - Enter your own code")
print("="*50)

while True:
    print("\nEnter a code snippet (or 'quit' to exit):")
    user_code = input()

    if user_code.lower() == 'quit':
        break

    if len(user_code.strip()) == 0:
        print("Please enter some code")
        continue

    prediction, neighbors = final_model.predict_with_neighbors(user_code)

    print(f"\nPredicted language: {prediction}")
    print("\nNearest matches in training data:")
    for i, (lang, dist, snippet) in enumerate(neighbors[:3], 1):
        print(f"  {i}. {lang} (distance={dist:.3f})")
        print(f"     Code: {snippet}...")

    features = extract_features(user_code)
    active_features = [name for name, val in features.items() if val]
    if active_features:
        print(f"\nDetected features: {', '.join(active_features[:5])}")

print("\nProject Results")
print(f"Best Accuracy: {best_accuracy*100:.2f}% using k={best_k}")
print(f"Training set: {len(X_train)} ({len(X_train)/len(df)*100:.1f}%)")
print(f"Testing set: {len(X_test)} ({len(X_test)/len(df)*100:.1f}%)")
print(f"Total Samples: {len(df)}")

Upload your CSV file


Saving code_dataset.csv to code_dataset.csv

Loaded 183 samples

Language distribution:
  Cpp: 64
  Python: 60
  HTML: 59

Training: 146 samples
Testing: 37 samples

Finding optimal K value...
  K=1: 94.6%
  K=3: 91.9%
  K=5: 91.9%
  K=7: 91.9%
  K=9: 91.9%
  K=11: 94.6%
  K=13: 94.6%
  K=15: 94.6%

Best K: 1 with 94.6% accuracy

Training final model...

Model Performance
Accuracy: 94.59%

Per-language metrics:
  Cpp:
    Precision: 92.3%
    Recall:    92.3%
    F1-Score:  92.3%
  HTML:
    Precision: 100.0%
    Recall:    100.0%
    F1-Score:  100.0%
  Python:
    Precision: 90.9%
    Recall:    90.9%
    F1-Score:  90.9%

Confusion Matrix (first 5x5):
  Actual HTML: {'HTML': 13, 'Python': 0, 'Cpp': 0}
  Actual Cpp: {'Cpp': 12, 'Python': 1, 'HTML': 0}
  Actual Python: {'Python': 10, 'Cpp': 1, 'HTML': 0}

Sample predictions on test data (first 10):
✓ Pred: HTML     | Actual: HTML     | <section>\n<article>Content</article>\n<...
✓ Pred: HTML     | Actual: HTML     | <title>Webpage Tit